# Reproducing Paper Results: Certificates and Verification

This notebook reproduces the key computational results from the paper and thesis:

1. Oracle enumeration yielding |F| = 1265
2. Verification of explicit feasible-potential certificates for ECP (5.4), (5.5), and Remark 5.5
3. Quadratic polynomial extraction and cross-check against enumerated rays
4. Copy-ready LaTeX tables for the thesis appendix

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "PercolationOracle"))
Pkg.develop(path=joinpath(@__DIR__, "..", "ProjectedConeOracle"))
Pkg.instantiate()

using PercolationOracle
using ProjectedConeOracle
using SparseArrays
using Printf

function enumerate_quiet(n_obs, m; kwargs...)
    result = redirect_stdout(devnull) do
        redirect_stderr(devnull) do
            enumerate_all_inequalities(n_obs, m; verbose=false, kwargs...)
        end
    end
    return result
end

println("Julia version: ", VERSION)
println("PercolationOracle: ", pathof(PercolationOracle))

  Activating 

project at `~/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle`


   Resolving 

package versions...


  No Changes to `~/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle/Project.toml`
  No Changes to `~/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle/Manifest.toml`


Julia version: 1

.11.6
PercolationOracle: /Users/azimin/Documents/Thesis/bunkbed-oracle-packages/PercolationOracle/src/PercolationOracle.jl


## 1. Notation

ECP partition notation: $\mathcal{J}_3 = \{abc,\; a|b|c,\; a|bc,\; ab|c,\; ac|b\}$

| ECP | Internal |
|-----|----------|
| abc | 123 |
| a\|b\|c | 1\|2\|3 |
| a\|bc | 1\|23 |
| ab\|c | 12\|3 |
| ac\|b | 13\|2 |

## 2. The Paper's Four-Tree Family

In [2]:
n_obs = 3

println("Decision trees (4 trees + 4 complements):")
for (i, label) in enumerate(PAPER_FAMILY_LABELS)
    println("  $i. $label")
end

println("\nPartition set J_$n_obs (B_$n_obs = $(bell_number(n_obs))):")
for (i, label) in enumerate(partition_order_labels(n_obs))
    println("  $i. $label")
end

println("\nThe tuple archive is normalized to paper order (T_0, T_1, T_2, T_3) on load.")

Decision trees (4 trees + 4 complements):


  1. T0
  2. T0_complement
  3. T1
  4. T1_complement
  5. T2
  6. T2_complement
  7. T3
  8. T3_complement

Partition set J_3 (B_3 = 5):
  1. 123


  2. 1|2|3
  3. 1|23
  4. 12|3
  5. 13|2

The tuple archive is normalized to paper order (T_0, T_1, T_2, T_3) on load.


## 3. Oracle Enumeration

In [3]:
result = enumerate_paper_family_regression()

println("Total |F| = $(result.count)")
println("Tuple hash: $(result.hash)")
println("\nPer-condensation-graph counts:")
for (pid, cnt) in zip(all_partition_ids(n_obs), result.counts)
    println("  A = $(partition_label(n_obs, pid)) => $cnt tuples")
end
println("\nSum: $(sum(result.counts))")

@assert result.count == 1265 "|F| must be 1265"
@assert result.counts == [25, 240, 240, 60, 700] "Per-graph counts must match paper"

Total |F| = 1265


Tuple hash: a7b38fd420c769e00b80659729b95c11b58dec3a

Per-condensation-graph counts:
  A = 123 => 25 tuples
  A = 12|3 => 240 tuples
  A = 13|2 => 240 tuples
  A = 1|23 => 60 tuples
  A = 1|2|3 => 700 tuples

Sum: 1265


## 4. Constraint Matrix

The feasible-potentials framework requires a constraint matrix $M_F$ with one row per feasible
tuple and columns indexed by $\varphi_k(p, \bar{p})$.
For $m=4$ trees and $|\mathcal{J}_3|=5$ partitions: $4 \times 5 \times 5 = 100$ variables.

In [4]:
F = paper_family_feasible_tuples()
m = 4

M = build_constraint_matrix(F, n_obs, m)

println("Constraint matrix M_F:")
println("  Size: $(size(M))  (rows = |F|, cols = m  x  B_3^2)")
println("  Nonzeros: $(nnz(M))")
println("  Nonzeros per row: $(nnz(M) / size(M, 1))")

@assert size(M) == (1265, 100)
@assert nnz(M) == 1265 * 4

Constraint matrix M_F:


  Size: (1265, 100)  (rows = |F|, cols = m  x  B_3^2)
  Nonzeros: 5060
  Nonzeros per row: 4.0


## 5. Certificate Sources

A **certificate** is a list of integer tables $\Phi^{(k)}\colon \mathcal{J}_3^2 \to \mathbb{Z}$
(one per tree) such that for every feasible tuple
$((p_0,\bar{p}_0),\ldots,(p_3,\bar{p}_3)) \in F$:
$\sum_k \Phi^{(k)}(p_k, \bar{p}_k) \geq 0$.
This is the dual feasibility condition from the feasible potentials framework.

Three certificates, loaded from the package:

| ECP label | Package function | Thesis label |
|-----------|------------------|--------------|
| (5.4) | `appendixA_certificate_11()` | Eq (11) |
| (5.5) | `appendixA_certificate_12()` | Eq (12) / Aas |
| Remark 5.5 | `appendixA_certificate_ray15()` | Ray 15 |

In [5]:
cert_54  = appendixA_certificate_11()    # ECP (5.4) = thesis Eq (11)
cert_55  = appendixA_certificate_12()    # ECP (5.5) = thesis Eq (12) = Aas
cert_r55 = appendixA_certificate_ray15() # ECP Remark 5.5 = thesis Ray 15

println("Loaded 3 certificates, each with $(length(cert_54)) blocks of size $(size(cert_54[1]))")

Loaded 3 certificates, each with 4 blocks of size (5, 5)


## 6. Certificate Verification

In [6]:
certs = [
    ("ECP (5.4)",    cert_54),
    ("ECP (5.5)",    cert_55),
    ("Remark 5.5",   cert_r55),
]

println("Certificate verification on F (|F| = $(length(F))):\n")
for (name, cert) in certs
    v = verify_certificate(cert, F; n_obs=n_obs, m=m)
    status = v.feasible ? "PASS" : "FAIL"
    println("  $name: $status  (min = $(v.minimum), negatives = $(v.negatives))")
    @assert v.feasible "$name certificate must be feasible"
    @assert v.minimum == 0 "$name minimum must be 0"
end

Certificate verification on F (|F| = 1265):

  ECP (5.4): PASS  (min = 0, negatives = 0)


  ECP (5.5): PASS  (min = 0, negatives = 0)
  Remark 5.5: PASS  (min = 0, negatives = 0)


## 7. Quadratic Polynomial Extraction

By Theorem 5.3, the aggregate $A(p, \bar{p}) = \sum_k \varphi_k(p, \bar{p})$ defines a
quadratic form in partition probabilities. The diagonal coefficient for $\mu(p)^2$ is $A(p,p)$,
and the off-diagonal coefficient for $\mu(p)\mu(q)$ (with $p \neq q$) is $A(p,q) + A(q,p)$.

In [7]:
# Internal -> ECP label translation
const TO_ECP = Dict(
    "123"=>"abc", "1|2|3"=>"a|b|c", "1|23"=>"a|bc", "12|3"=>"ab|c", "13|2"=>"ac|b"
)
ecp(s) = TO_ECP[s]

function display_polynomial(poly, name)
    println("$name  --  quadratic form coefficients:\n")
    
    # Diagonal
    has_diag = false
    for pid in poly.order
        c = poly.diagonal[pid]
        c == 0 && continue
        has_diag = true
        println("    mu($(ecp(partition_label(n_obs, pid))))^2:  $c")
    end
    has_diag || println("    (no diagonal terms)")
    
    # Off-diagonal
    println()
    for i in eachindex(poly.order), j in i+1:length(poly.order)
        pi, pj = poly.order[i], poly.order[j]
        c = poly.offdiag[(pi, pj)]
        c == 0 && continue
        li = ecp(partition_label(n_obs, pi))
        lj = ecp(partition_label(n_obs, pj))
        println("    mu($li)*mu($lj):  $c")
    end
    println()
end

for (name, cert) in certs
    poly = extract_quadratic_polynomial(cert, n_obs)
    display_polynomial(poly, name)
end

ECP (5.4)  --  quadratic form coefficients:



    mu(a|bc)^2:  1

    mu(abc)*mu(a|b|c):  -1
    mu(abc)*mu(ab|c):  1
    mu(abc)*mu(ac|b):  1
    mu(a|b|c)*mu(a|bc):  1
    mu(a|bc)*mu(ab|c):  1
    mu(a|bc)*mu(ac|b):  1
    mu(ab|c)*mu(ac|b):  2

ECP (5.5)  --  quadratic form coefficients:

    (no diagonal terms)

    mu(abc)*mu(a|b|c):  1
    mu(a|bc)*mu(ab|c):  -1
    mu(a|bc)*mu(ac|b):  -1
    mu(ab|c)*mu(ac|b):  -1

Remark 5.5  --  quadratic form coefficients:

    (no diagonal terms)

    mu(abc)*mu(a|b|c):  -1
    mu(abc)*mu(a|bc):  1
    mu(abc)*mu(ab|c):  1
    mu(abc)*mu(ac|b):  1
    mu(a|b|c)*mu(a|bc):  1
    mu(a|bc)*mu(ab|c):  1
    mu(a|bc)*mu(ac|b):  1
    mu(ab|c)*mu(ac|b):  2



## 8. Cross-Check Against Enumerated Rays

The strongest correctness check: each certificate's polynomial signature must match an
independently enumerated extreme ray from the D1 $\to$ D2 pipeline.

In [8]:
result_m4 = enumerate_quiet(3, 4)
@assert length(result_m4.rays) == 17 "Expected 17 extreme rays"

println("Cross-checking certificates against $(length(result_m4.rays)) enumerated rays:\n")

# Internal -> ECP for formatted inequalities display
function to_ecp_str(s::AbstractString)
    r = s
    for (from, to) in sort(collect(TO_ECP), by=x->length(x[1]), rev=true)
        r = replace(r, from => to)
    end
    return r
end

for (name, cert) in certs
    sig = polynomial_signature(cert, n_obs)
    matched = false
    for (i, ray) in enumerate(result_m4.rays)
        ray_sig = canonicalize_integer_ray(ray; normalize_sign=false)
        if ray_sig == sig
            ecp_ineq = to_ecp_str(result_m4.formatted_inequalities[i])
            println("  $name => ray #$i")
            println("    $ecp_ineq")
            println()
            matched = true
            break
        end
    end
    @assert matched "$name must match an enumerated ray"
end

Cross-checking certificates against 17 enumerated rays:

  ECP (5.4) => ray #16


    mu(a|bc)^2 - mu(abc)*mu(a|b|c) + mu(abc)*mu(ab|c) + mu(abc)*mu(ac|b) + mu(a|b|c)*mu(a|bc) + mu(a|bc)*mu(ab|c) + mu(a|bc)*mu(ac|b) + 2*mu(ab|c)*mu(ac|b) >= 0

  ECP (5.5) => ray #1
    mu(abc)*mu(a|b|c) - mu(a|bc)*mu(ab|c) - mu(a|bc)*mu(ac|b) - mu(ab|c)*mu(ac|b) >= 0

  Remark 5.5 => ray #15
    -mu(abc)*mu(a|b|c) + mu(abc)*mu(a|bc) + mu(abc)*mu(ab|c) + mu(abc)*mu(ac|b) + mu(a|b|c)*mu(a|bc) + mu(a|bc)*mu(ab|c) + mu(a|bc)*mu(ac|b) + 2*mu(ab|c)*mu(ac|b) >= 0



## 9. LaTeX Tables for Thesis Appendix

The tables below can be pasted directly into
`chapter_bond_oracle/sections/A_appendix_certificates.tex`.
Each certificate produces 4 blocks $\Phi^{(1)},\ldots,\Phi^{(4)}$ with rows/columns in paper
order (abc, a|b|c, a|bc, ab|c, ac|b).

In [9]:
function emit_latex_tables(cert_blocks, label_prefix)
    ecp_labels = ["abc", "a|b|c", "a|bc", "ab|c", "ac|b"]
    for (k, block) in enumerate(cert_blocks)
        is_zero = all(iszero, block)
        zero_note = is_zero ? " (identically zero)" : ""
        
        println("\\begin{table}[ht]")
        println("\\centering")
        println("\\small")
        println("\\setlength{\\tabcolsep}{5pt}")
        println("\\renewcommand{\\arraystretch}{1.1}")
        println("\\begin{tabular}{r|rrrrr}")
        print(" & ")
        println(join(["\\($l\\)" for l in ecp_labels], " & "), " \\\\\\hline")
        for (i, rl) in enumerate(ecp_labels)
            vals = join([@sprintf("%2d", block[i,j]) for j in 1:5], " & ")
            println("\\($(rpad(rl * "\\)", 8)) & $vals \\\\")
        end
        println("\\end{tabular}")
        println("\\caption{Block \\(\\Phi^{($k)}\\) for inequality~\\eqref{$label_prefix}$zero_note.}")
        println("\\label{bo:tab:$(replace(label_prefix, "bo:eq:"=>""))-block$k}")
        println("\\end{table}")
        println()
    end
end

# Get certificates in paper order for LaTeX
cert_54_paper  = appendixA_certificate_11(order=:paper)
cert_55_paper  = appendixA_certificate_12(order=:paper)
cert_r55_paper = appendixA_certificate_ray15(order=:paper)

# Inequality LaTeX for rendering in notebook
INEQ_LATEX = Dict(
    "bo:eq:ineq11" => raw"$\mu(a|bc)^2 + \mu(abc)\mu(ab|c) + \mu(abc)\mu(ac|b) - \mu(abc)\mu(a|b|c) + \mu(a|bc)\mu(a|b|c) + \mu(a|bc)\mu(ab|c) + \mu(a|bc)\mu(ac|b) + 2\mu(ab|c)\mu(ac|b) \geq 0$",
    "bo:eq:ineq12" => raw"$\mu(abc)\mu(a|b|c) - \mu(a|bc)\mu(ab|c) - \mu(a|bc)\mu(ac|b) - \mu(ab|c)\mu(ac|b) \geq 0$",
    "bo:eq:ray15" => raw"$\mu(abc)\mu(ab|c) + \mu(abc)\mu(ac|b) + \mu(abc)\mu(a|bc) - \mu(abc)\mu(a|b|c) + 2\mu(ab|c)\mu(ac|b) + \mu(ab|c)\mu(a|bc) + \mu(ac|b)\mu(a|bc) + \mu(a|bc)\mu(a|b|c) \geq 0$",
)

latex_specs = [
    ("ECP (5.4) / Equation (11)", cert_54_paper, "bo:eq:ineq11"),
    ("ECP (5.5) / Equation (12) / Aas", cert_55_paper, "bo:eq:ineq12"),
    ("ECP Remark 5.5 / Ray 15", cert_r55_paper, "bo:eq:ray15"),
]

for (name, blocks, prefix) in latex_specs
    # Render the inequality as LaTeX in notebook
    display(MIME"text/latex"(), "\\textbf{$name:}\\quad " * INEQ_LATEX[prefix][2:end])

    # Print raw TeX tables for copy-paste
    println("\n% ========== $name ==========\n")
    emit_latex_tables(blocks, prefix)
    println()
end




% ========== ECP (5.4) / Equation (11) ==========

\begin{table}[ht]


\centering
\small
\setlength{\tabcolsep}{5pt}
\renewcommand{\arraystretch}{1.1}
\begin{tabular}{r|rrrrr}
 & \(abc\) & \(a|b|c\) & \(a|bc\) & \(ab|c\) & \(ac|b\) \\\hline
\(abc\)    &  0 &  0 &  0 &  0 &  0 \\
\(a|b|c\)  &  0 &  0 &  0 &  0 &  0 \\
\(a|bc\)   &  0 &  0 &  0 &  0 &  0 \\
\(ab|c\)   &  0 &  0 &  0 &  0 &  0 \\
\(ac|b\)   &  0 &  0 &  0 &  0 &  0 \\
\end{tabular}
\caption{Block \(\Phi^{(1)}\) for inequality~\eqref{bo:eq:ineq11} (identically zero).}
\label{bo:tab:ineq11-block1}
\end{table}

\begin{table}[ht]
\centering
\small
\setlength{\tabcolsep}{5pt}
\renewcommand{\arraystretch}{1.1}
\begin{tabular}{r|rrrrr}
 & \(abc\) & \(a|b|c\) & \(a|bc\) & \(ab|c\) & \(ac|b\) \\\hline
\(abc\)    & -1 &  0 &  0 & -1 &  0 \\
\(a|b|c\)  &  0 &  0 &  0 &  0 &  0 \\
\(a|bc\)   &  1 &  0 &  0 &  1 &  0 \\
\(ab|c\)   &  1 &  1 &  1 &  1 &  1 \\
\(ac|b\)   &  0 &  0 &  0 &  0 &  0 \\
\end{tabular}
\caption{Block \(\Phi^{(2)}\) for inequality~\eqref{bo:eq:ineq11}.}
\label{bo:tab:ineq11-block2


% ========== ECP (5.5) / Equation (12) / Aas ==========

\begin{table}[ht]
\centering
\small
\setlength{\tabcolsep}{5pt}
\renewcommand{\arraystretch}{1.1}
\begin{tabular}{r|rrrrr}
 & \(abc\) & \(a|b|c\) & \(a|bc\) & \(ab|c\) & \(ac|b\) \\\hline
\(abc\)    &  0 &  0 &  0 &  0 &  0 \\
\(a|b|c\)  &  0 &  0 &  0 &  0 &  0 \\
\(a|bc\)   &  0 &  0 &  0 &  0 &  0 \\
\(ab|c\)   &  0 &  0 &  0 &  0 &  0 \\
\(ac|b\)   &  0 &  0 &  0 &  0 &  0 \\
\end{tabular}
\caption{Block \(\Phi^{(1)}\) for inequality~\eqref{bo:eq:ineq12} (identically zero).}
\label{bo:tab:ineq12-block1}
\end{table}

\begin{table}[ht]
\centering
\small
\setlength{\tabcolsep}{5pt}
\renewcommand{\arraystretch}{1.1}
\begin{tabular}{r|rrrrr}
 & \(abc\) & \(a|b|c\) & \(a|bc\) & \(ab|c\) & \(ac|b\) \\\hline
\(abc\)    &  0 &  0 &  0 &  0 &  0 \\
\(a|b|c\)  &  1 &  1 &  1 &  1 &  1 \\
\(a|bc\)   &  0 &  0 &  0 &  0 &  0 \\
\(ab|c\)   &  0 &  0 &  0 &  0 &  0 \\
\(ac|b\)   &  0 &  0 &  0 &  0 &  0 \\
\end{tabular}
\caption{Block \(\P


% ========== ECP Remark 5.5 / Ray 15 ==========

\begin{table}[ht]
\centering
\small
\setlength{\tabcolsep}{5pt}
\renewcommand{\arraystretch}{1.1}
\begin{tabular}{r|rrrrr}
 & \(abc\) & \(a|b|c\) & \(a|bc\) & \(ab|c\) & \(ac|b\) \\\hline
\(abc\)    &  0 &  0 &  0 &  0 &  0 \\
\(a|b|c\)  &  0 &  0 &  0 &  0 &  0 \\
\(a|bc\)   &  0 &  0 &  0 &  0 &  0 \\
\(ab|c\)   &  0 &  0 &  0 &  0 &  0 \\
\(ac|b\)   &  0 &  0 &  0 &  0 &  0 \\
\end{tabular}
\caption{Block \(\Phi^{(1)}\) for inequality~\eqref{bo:eq:ray15} (identically zero).}
\label{bo:tab:ray15-block1}
\end{table}

\begin{table}[ht]
\centering
\small
\setlength{\tabcolsep}{5pt}
\renewcommand{\arraystretch}{1.1}
\begin{tabular}{r|rrrrr}
 & \(abc\) & \(a|b|c\) & \(a|bc\) & \(ab|c\) & \(ac|b\) \\\hline
\(abc\)    &  0 &  0 &  0 &  0 &  0 \\
\(a|b|c\)  &  0 &  0 &  0 &  0 &  0 \\
\(a|bc\)   &  1 &  0 &  0 &  0 &  1 \\
\(ab|c\)   &  1 &  1 &  0 &  1 &  1 \\
\(ac|b\)   &  0 &  0 &  0 &  0 &  0 \\
\end{tabular}
\caption{Block \(\Phi^{(2)}\)

## 10. Summary

In [10]:
# Gather all results
reg = enumerate_paper_family_regression()
v_54 = verify_certificate(cert_54, F; n_obs=n_obs, m=m)
v_55 = verify_certificate(cert_55, F; n_obs=n_obs, m=m)
v_r55 = verify_certificate(cert_r55, F; n_obs=n_obs, m=m)

# Check ray matches (reuse result_m4 from above)
function ray_match(cert)
    sig = polynomial_signature(cert, n_obs)
    for (i, ray) in enumerate(result_m4.rays)
        canonicalize_integer_ray(ray; normalize_sign=false) == sig && return "ray #$i"
    end
    return "NO MATCH"
end

items = [
    ("|F|",                "1265",    string(reg.count)),
    ("counts",             "[25,240,240,60,700]", string(reg.counts)),
    ("rays",               "17",      string(length(result_m4.rays))),
    ("(5.4) feasible",     "yes",     v_54.feasible ? "yes (min=$(v_54.minimum))" : "NO"),
    ("(5.5) feasible",     "yes",     v_55.feasible ? "yes (min=$(v_55.minimum))" : "NO"),
    ("Remark 5.5 feasible","yes",     v_r55.feasible ? "yes (min=$(v_r55.minimum))" : "NO"),
    ("(5.4) matches ray",  "yes",     ray_match(cert_54)),
    ("(5.5) matches ray",  "yes",     ray_match(cert_55)),
    ("Remark 5.5 matches", "yes",     ray_match(cert_r55)),
]

println("Final verification summary:\n")
println(rpad("Item", 24), rpad("Expected", 22), "Observed")
println("-"^68)
for (item, expected, observed) in items
    println(rpad(item, 24), rpad(expected, 22), observed)
end

Final verification summary:

Item                    Expected              Observed
--------------------------------------------------------------------
|F|                     1265                  1265
counts                  [25,240,240,60,700]   

[25, 240, 240, 60, 700]
rays                    17                    17
(5.4) feasible          yes                   yes (min=0)
(5.5) feasible          yes                   yes (min=0)
Remark 5.5 feasible     yes                   yes (min=0)
(5.4) matches ray       yes                   ray #16
(5.5) matches ray       yes                   ray #1
Remark 5.5 matches      yes                   ray #15
